# Is Hier-COS inference itself the main advantage?

This notebook performs an inference-only, paired comparison on existing checkpoints. No model is retrained.

Every inference rule is one cell of a **readout × transform** grid, and the same four cells are evaluated for every model:

| | `node_score` | `subspace_norm` |
|---|---|---|
| **no transform** | rank each node by its own coordinate | rank it by the L2 norm over its ancestors+self+descendants subspace |
| **HCC projection** | `hcc_node_score` | `hcc_subspace_norm` |

Each checkpoint's own inference is one of these cells — `node_score` for H-CAST and HRN, `subspace_norm` for Hier-COS — so the comparison asks one question of every model: **does aggregating over the taxonomy subspace, or projecting onto the hierarchy constraint, beat what the model already does?**

The key quantity is the within-checkpoint **gain against that model's native cell**. Positive always means the listed cell is better, including for lower-is-better AHD and TICE. Absolute cross-model scores are secondary because architectures and training objectives differ. Every completed seed is evaluated separately; its YAML stays in that seed's run directory, and tables/plots report mean ± sample standard deviation with the seed count.

Use the **WHAT TO SHOW** cell to pick which inference cells, decoder (independent, top-down, or both), datasets, models, and metrics to display. Everything below reads that selection, so no reload is needed to change the view.

## Experimental cautions

1. H-CAST/HRN logits were not trained as Hier-COS coordinates; identity-frame decoding is a post-hoc assumption.
2. The available best checkpoints were selected using each model's native validation inference. This can favor native inference.
3. Top-down and independent rows must use their corresponding best checkpoints. This notebook preserves that rule.
4. Unequal seed coverage must be reported. A one-seed sign is suggestive, not robust evidence.
5. Treat this test-set analysis as a pre-specified ablation. Do not tune normalization or score variants repeatedly on test data.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import torch
import yaml

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'evaluation').is_dir():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'evaluation').is_dir(), 'Run this notebook from the repository or notebooks directory.'

NOTEBOOK_UTILS = str(REPO_ROOT / 'notebooks')
if NOTEBOOK_UTILS not in sys.path:
    sys.path.insert(0, NOTEBOOK_UTILS)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from multiseed_utils import discover_seed_dirs, sample_stats

OUTPUTS_ROOT = Path('/scratch/g.saggini1/outputs')
DATASETS = ('cifar100', 'cub200', 'aircraft')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    print('CUDA is unavailable; inference will run on CPU and may be slow.')
RUN_EVALUATION = True  # The next execution cell runs the Python checkpoint evaluator.
OVERWRITE = False
# 'all' evaluates the four readout x transform cells; 'both' only the two
# untransformed readouts. Existing YAMLs are reused as they are, so set
# OVERWRITE=True to regenerate older two-cell files with all four.
INFERENCE_MODE = 'all'

BASELINE_ROOTS = {
    'cifar100': {
        'hcast': OUTPUTS_ROOT / 'hcast_cifar100',
        #'hrn': OUTPUTS_ROOT / 'hrn_cifar100',
        'hiercos': OUTPUTS_ROOT / 'hiercos_cifar100_global_softmax_ce_reg_baseline_kl_leaf',
    },
    'cub200': {
        'hcast': OUTPUTS_ROOT / 'hcast_cub200',
        #'hrn': OUTPUTS_ROOT / 'hrn_cub200',
        'hiercos': OUTPUTS_ROOT / 'hiercos_cub200_global_softmax_ce_reg_baseline_kl_leaf',
    },
    'aircraft': {
        'hcast': OUTPUTS_ROOT / 'hcast_aircraft',
        'hrn': OUTPUTS_ROOT / 'hrn_aircraft',
        'hiercos': OUTPUTS_ROOT / 'hiercos_aircraft_global_softmax_ce_reg_baseline_kl_leaf',
    },
}

BASELINE_ROOTS

In [14]:
def completed_seed_dirs(root: Path):
    seed_dirs = discover_seed_dirs(root, require_log=False, require_config=True)
    candidates = seed_dirs or ([root] if root.is_dir() else [])
    required = ('config_resolved.yaml', 'best_topdown.pt', 'best_independent.pt')
    return [path for path in candidates if all((path / name).is_file() for name in required)]

run_dirs = {
    dataset: {model: completed_seed_dirs(root) for model, root in BASELINE_ROOTS[dataset].items()}
    for dataset in DATASETS
}
coverage = pd.DataFrame(
    [
        {'dataset': dataset, 'model': model, 'runs': len(paths), 'paths': [str(path) for path in paths]}
        for dataset, model_runs in run_dirs.items()
        for model, paths in model_runs.items()
    ]
)
display(coverage)
missing_baselines = coverage.loc[coverage['runs'].eq(0), ['dataset', 'model']]
if not missing_baselines.empty:
    print('Missing completed baselines:')
    display(missing_baselines)
    print('Edit BASELINE_ROOTS or finish those runs before making a three-model claim.')

,dataset,model,runs,paths
0,cifar100,hcast,3,[/scratch/g.saggini1/outputs/hcast_cifar100/se...
1,cifar100,hiercos,3,[/scratch/g.saggini1/outputs/hiercos_cifar100_...
2,cub200,hcast,3,[/scratch/g.saggini1/outputs/hcast_cub200/seed...
3,cub200,hiercos,3,[/scratch/g.saggini1/outputs/hiercos_cub200_gl...
4,aircraft,hcast,3,[/scratch/g.saggini1/outputs/hcast_aircraft/se...
5,aircraft,hrn,3,[/scratch/g.saggini1/outputs/hrn_aircraft/seed...
6,aircraft,hiercos,3,[/scratch/g.saggini1/outputs/hiercos_aircraft_...


In [ ]:
result_files = []
for dataset, model_runs in run_dirs.items():
    for model, paths in model_runs.items():
        for run_dir in paths:
            output_path = run_dir / 'posthoc_inference_test_metrics.yaml'
            result_files.append(output_path)
            if not RUN_EVALUATION:
                continue
            if output_path.exists() and not OVERWRITE:
                print('[reuse]', output_path)
                continue
            command = [
                sys.executable, '-m', 'evaluation.evaluate_checkpoints',
                '--run-dir', str(run_dir),
                '--inference-mode', INFERENCE_MODE,
                '--checkpoint-mode', 'both',
                '--device', DEVICE,
            ]
            if OVERWRITE:
                command.append('--overwrite')
            print('[run]', ' '.join(command))
            subprocess.run(command, cwd=REPO_ROOT, check=True)

available_result_files = [path for path in result_files if path.is_file()]
print(f'Available result files: {len(available_result_files)}/{len(result_files)}')
if not RUN_EVALUATION and not available_result_files:
    print('Set RUN_EVALUATION=True and run this cell to generate results.')

# Files written before the four-cell grid existed hold only two inference rows.
# They are still readable — the loader maps their names onto the grid — but the
# missing cells simply will not be selectable until they are regenerated.
partial = []
fallback = []
for path in available_result_files:
    payload = yaml.safe_load(path.read_text())
    if len(payload.get('resolved_inference_modes', [])) < 4:
        partial.append(path)
    if payload.get('test_split_source') == 'official_dataset_adapter_fallback':
        fallback.append(path)
if partial:
    print(f'\n{len(partial)}/{len(available_result_files)} files predate the four-cell grid '
          f'or were written with --inference-mode both.')
    print('Set OVERWRITE=True and re-run this cell to fill in the missing cells.')
if fallback:
    print(f'\n[warning] {len(fallback)} file(s) were evaluated with the official-adapter '
          'fallback because the run\'s configured test manifest is missing. Their label '
          'space may differ from what the run trained on; check the native row against '
          'the run\'s own test_metrics.yaml before trusting them:')
    for path in fallback:
        print('   ', path)

In [ ]:
from evaluation.evaluate_checkpoints import canonical_inference_rule, native_inference_rule

DECODERS = ('independent', 'topdown')


def split_metric(metric):
    """Return (metric_family, decoder) for one metric key, or (None, None)."""
    for decoder in DECODERS:
        if metric.startswith(f'acc_level_{decoder}_'):
            return f"acc_level_{metric.rsplit('_', 1)[-1]}", decoder
        if metric.endswith(f'_{decoder}'):
            return metric[: -(len(decoder) + 1)], decoder
    return None, None


def native_cell_of(payload):
    """The grid cell that reproduces this checkpoint's own inference."""
    return payload.get('native_inference_mode') or native_inference_rule(
        payload['model'], bool(payload.get('hcc_trained_run', False))
    )


def canonical_cell(row_name, payload):
    """Map one YAML row onto a grid cell, for files written before or after the rename.

    `normal` named whatever the checkpoint did natively, so it follows the model;
    every other legacy name maps through the CLI's own alias table.
    """
    if row_name == 'normal':
        return native_cell_of(payload)
    return canonical_inference_rule(row_name, payload['model'])


# Every decoder is loaded. Selection happens in the next cell, so switching
# between independent, top-down, or both never needs a reload.
records = []
for result_path in available_result_files:
    payload = yaml.safe_load(result_path.read_text())
    native_cell = native_cell_of(payload)
    for checkpoint_mode, checkpoint_payload in payload['checkpoints'].items():
        for row_name, metrics in checkpoint_payload['inference'].items():
            inference = canonical_cell(row_name, payload)
            for metric, value in metrics.items():
                metric_family, decoder = split_metric(metric)
                if metric_family is None:
                    continue
                records.append({
                    'dataset': payload['dataset'],
                    'model': payload['model'],
                    'seed': int(payload['seed']),
                    'checkpoint_mode': checkpoint_mode,
                    'decoder': decoder,
                    'inference': inference,
                    'is_native': inference == native_cell,
                    'metric_family': metric_family,
                    'metric': metric,
                    'value': float(value),
                })

results_long = pd.DataFrame(records)
if results_long.empty:
    print('No result rows loaded yet.')
else:
    coverage_cells = (
        results_long.groupby(['dataset', 'model'])['inference']
        .agg(lambda values: ', '.join(sorted(set(values))))
        .reset_index()
        .rename(columns={'inference': 'available inference cells'})
    )
    print('Loaded rows:', len(results_long))
    display(coverage_cells)

In [ ]:
# =====================  WHAT TO SHOW  =====================================
# Edit these, then re-run this cell and the ones below.
SHOW_INFERENCES = 'all'      # 'all' | 'native' | e.g. ['node_score', 'hcc_node_score']
SHOW_DECODERS = 'both'       # 'both' | 'independent' | 'topdown'
SHOW_DATASETS = 'all'        # 'all' | e.g. ['cifar-100', 'fgvc-aircraft']
SHOW_MODELS = 'all'          # 'all' | e.g. ['hcast', 'hiercos']
SHOW_METRICS = ('fpa', 'weighted_ap', 'tice', 'ahd')   # add 'acc_level_0', 'acc_level_1', ...
MATCHED_CHECKPOINT = True    # repo rule: read each decoder from its own selected checkpoint
# ==========================================================================

INFERENCE_ORDER = ('node_score', 'subspace_norm', 'hcc_node_score', 'hcc_subspace_norm')
INFERENCE_LABELS = {
    'node_score': 'node_score',
    'subspace_norm': 'subspace_norm',
    'hcc_node_score': 'hcc + node_score',
    'hcc_subspace_norm': 'hcc + subspace_norm',
}
INFERENCE_COLORS = {
    'node_score': '#2563eb',
    'subspace_norm': '#16a34a',
    'hcc_node_score': '#d97706',
    'hcc_subspace_norm': '#7c3aed',
}
DECODER_LABELS = {'independent': 'Independent', 'topdown': 'Top-down'}


def _as_list(value, available):
    """Normalize a selector into a concrete list of values that exist in the data."""
    if value is None or (isinstance(value, str) and value == 'all'):
        return list(available)
    if isinstance(value, str):
        value = [value]
    unknown = [item for item in value if item not in available]
    if unknown:
        print(f'[note] not present in the loaded data, ignored: {unknown}')
    return [item for item in value if item in available]


def select(
    frame=None,
    inferences=None,
    decoders=None,
    datasets=None,
    models=None,
    metrics=None,
    matched_checkpoint=None,
):
    """Return the subset of rows to display.

    Defaults come from the SHOW_* settings above, so `select()` gives the current
    view and `select(decoders='topdown', inferences=['subspace_norm'])` answers a
    one-off question without touching them.

    `matched_checkpoint=True` keeps the repository rule that a decoder is read
    from its own validation-selected checkpoint. Setting it False also shows the
    crossed combinations, which are not comparable to the reported results.
    """
    frame = results_long if frame is None else frame
    if frame.empty:
        return frame

    inferences = SHOW_INFERENCES if inferences is None else inferences
    decoders = SHOW_DECODERS if decoders is None else decoders
    datasets = SHOW_DATASETS if datasets is None else datasets
    models = SHOW_MODELS if models is None else models
    metrics = SHOW_METRICS if metrics is None else metrics
    matched = MATCHED_CHECKPOINT if matched_checkpoint is None else matched_checkpoint

    selected = frame
    if isinstance(inferences, str) and inferences == 'native':
        selected = selected[selected['is_native']]
    else:
        available = [cell for cell in INFERENCE_ORDER if cell in set(frame['inference'])]
        selected = selected[selected['inference'].isin(_as_list(inferences, available))]

    decoders = ['independent', 'topdown'] if decoders == 'both' else decoders
    selected = selected[selected['decoder'].isin(_as_list(decoders, DECODERS))]
    selected = selected[selected['dataset'].isin(_as_list(datasets, sorted(set(frame['dataset']))))]
    selected = selected[selected['model'].isin(_as_list(models, sorted(set(frame['model']))))]
    selected = selected[
        selected['metric_family'].isin(_as_list(metrics, sorted(set(frame['metric_family']))))
    ]
    if matched:
        selected = selected[selected['decoder'].eq(selected['checkpoint_mode'])]
    return selected.reset_index(drop=True)


view = select()
if view.empty:
    print('Nothing selected. Widen the SHOW_* settings above.')
else:
    print(f'Selected {len(view)} rows')
    print('  inference cells :', ', '.join(c for c in INFERENCE_ORDER if c in set(view['inference'])))
    print('  decoders        :', ', '.join(sorted(set(view['decoder']))))
    print('  datasets        :', ', '.join(sorted(set(view['dataset']))))
    print('  models          :', ', '.join(sorted(set(view['model']))))
    print('  metrics         :', ', '.join(sorted(set(view['metric_family']))))
    print('  checkpoint rule :', 'matched to decoder' if MATCHED_CHECKPOINT else 'ALL (includes crossed pairs)')
    display(view.head())

In [ ]:
LOWER_IS_BETTER = ('tice', 'ahd')
PAIR_KEYS = ['dataset', 'model', 'seed', 'checkpoint_mode', 'decoder', 'metric_family', 'metric']


def display_value(frame, column='value'):
    """AHD stays in raw units; every other metric is a percentage."""
    return np.where(frame['metric_family'].eq('ahd'), frame[column], 100.0 * frame[column])


def summarize_samples(frame, group_columns, value_column, mean_column, std_column):
    rows = []
    for keys, group in frame.groupby(group_columns, dropna=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        mean, std, count = sample_stats(group[value_column].tolist())
        rows.append({
            **dict(zip(group_columns, keys)),
            mean_column: mean,
            std_column: std,
            'seeds': count,
        })
    return pd.DataFrame(rows)


def paired_gains(frame=None, reference=None):
    """Gain of each selected cell against the same checkpoint's native cell.

    Positive always means the selected cell is better, including for the
    lower-is-better AHD and TICE. The native row is taken from the full loaded
    table, so deselecting it does not remove the reference.
    """
    frame = view if frame is None else frame
    reference = results_long if reference is None else reference
    if frame.empty or reference.empty:
        return pd.DataFrame()

    native = (
        reference[reference['is_native']][PAIR_KEYS + ['value', 'inference']]
        .rename(columns={'value': 'native_value', 'inference': 'native_inference'})
    )
    merged = frame[~frame['is_native']].merge(native, on=PAIR_KEYS, how='inner')
    if merged.empty:
        return merged
    sign = np.where(merged['metric_family'].isin(LOWER_IS_BETTER), -1.0, 1.0)
    merged['gain'] = sign * (merged['value'] - merged['native_value'])
    merged['gain_display'] = np.where(
        merged['metric_family'].eq('ahd'), merged['gain'], 100.0 * merged['gain']
    )
    return merged


if not view.empty:
    view = view.copy()
    view['value_display'] = display_value(view)
    absolute_summary = summarize_samples(
        view,
        ['dataset', 'model', 'decoder', 'inference', 'is_native', 'metric_family'],
        'value_display',
        'mean_value',
        'std_value',
    )
    print('Absolute results (mean, sample standard deviation, seed count):')
    display(absolute_summary)

    paired = paired_gains()
    if paired.empty:
        print('\nNo paired gains: only the native cell is selected, or its row is missing.')
        gain_summary = pd.DataFrame()
    else:
        gain_summary = summarize_samples(
            paired,
            ['dataset', 'model', 'decoder', 'inference', 'metric_family'],
            'gain_display',
            'mean_gain',
            'std_gain',
        )
        native_names = ', '.join(sorted(set(paired['native_inference'])))
        print(f'\nPaired gains against each checkpoint\'s native cell ({native_names}).')
        print('Positive favors the listed cell, for every metric.')
        display(gain_summary)
else:
    absolute_summary = pd.DataFrame()
    paired = pd.DataFrame()
    gain_summary = pd.DataFrame()

In [ ]:
MODEL_ORDER = ('hcast', 'hrn', 'hiercos')
MODEL_LABELS = {'hcast': 'H-CAST', 'hrn': 'HRN', 'hiercos': 'Hier-COS'}
DATASET_LABELS = {
    'cifar100': 'CIFAR-100', 'cifar-100': 'CIFAR-100',
    'cub200': 'CUB-200-2011', 'cub-200-2011': 'CUB-200-2011',
    'aircraft': 'FGVC-Aircraft', 'fgvc-aircraft': 'FGVC-Aircraft',
}
METRIC_SPECS = {
    'fpa': ('Full-path accuracy (FPA)', 'percentage points', True),
    'weighted_ap': ('Weighted per-level accuracy', 'percentage points', True),
    'tice': ('Taxonomy inconsistency (TICE)', 'percentage points', False),
    'ahd': ('Average hierarchy distance (AHD)', 'raw AHD units', False),
    'acc_level_0': ('Coarse accuracy', 'percentage points', True),
    'acc_level_1': ('Middle accuracy', 'percentage points', True),
    'acc_level_2': ('Fine accuracy', 'percentage points', True),
}
PLOT_OUTPUT_DIR = OUTPUTS_ROOT / 'analysis' / 'posthoc_hiercos_inference_comparison'
SAVE_PLOTS = True


def finite_std(row, column):
    value = row[column]
    return float(value) if row['seeds'] > 1 and pd.notna(value) else None


def plot_inference_comparison(dataset, decoder, absolute_table, gain_table, seed_table):
    """One row per metric: absolute means on the left, gain against native on the right."""
    absolute_slice = absolute_table[
        absolute_table['dataset'].eq(dataset) & absolute_table['decoder'].eq(decoder)
    ]
    if absolute_slice.empty:
        return

    metrics = [name for name in METRIC_SPECS if name in set(absolute_slice['metric_family'])]
    models = [model for model in MODEL_ORDER if model in set(absolute_slice['model'])]
    models += [model for model in sorted(set(absolute_slice['model'])) if model not in models]
    cells = [cell for cell in INFERENCE_ORDER if cell in set(absolute_slice['inference'])]
    if not metrics or not models or not cells:
        return

    gain_slice = (
        gain_table[gain_table['dataset'].eq(dataset) & gain_table['decoder'].eq(decoder)]
        if not gain_table.empty else gain_table
    )
    seed_slice = (
        seed_table[seed_table['dataset'].eq(dataset) & seed_table['decoder'].eq(decoder)]
        if not seed_table.empty else seed_table
    )

    y_positions = np.arange(len(models))
    spread = 0.30 if len(cells) > 1 else 0.0
    offsets = np.linspace(-spread, spread, len(cells)) if len(cells) > 1 else np.array([0.0])
    row_height = max(3.0, 0.35 * len(cells) * len(models))
    fig, axes = plt.subplots(
        len(metrics), 2, figsize=(15, row_height * len(metrics)), squeeze=False
    )

    for row_index, metric_name in enumerate(metrics):
        title, unit, higher_is_better = METRIC_SPECS[metric_name]
        absolute_ax, gain_ax = axes[row_index]
        gain_extent = [0.0]

        for cell, offset in zip(cells, offsets):
            color = INFERENCE_COLORS[cell]
            for y_position, model in zip(y_positions, models):
                absolute_rows = absolute_slice[
                    absolute_slice['metric_family'].eq(metric_name)
                    & absolute_slice['model'].eq(model)
                    & absolute_slice['inference'].eq(cell)
                ]
                if not absolute_rows.empty:
                    row = absolute_rows.iloc[0]
                    is_native = bool(row['is_native'])
                    absolute_ax.errorbar(
                        row['mean_value'], y_position + offset,
                        xerr=finite_std(row, 'std_value'),
                        fmt='o' if is_native else 'D', markersize=7, color=color,
                        markerfacecolor='white' if is_native else color,
                        markeredgewidth=1.5, capsize=3, linewidth=1.5, zorder=3,
                    )

                if gain_slice.empty:
                    continue
                gain_rows = gain_slice[
                    gain_slice['metric_family'].eq(metric_name)
                    & gain_slice['model'].eq(model)
                    & gain_slice['inference'].eq(cell)
                ]
                seed_rows = seed_slice[
                    seed_slice['metric_family'].eq(metric_name)
                    & seed_slice['model'].eq(model)
                    & seed_slice['inference'].eq(cell)
                ] if not seed_slice.empty else seed_slice
                if not seed_rows.empty:
                    seed_values = seed_rows['gain_display'].to_numpy()
                    jitter = np.linspace(-0.05, 0.05, len(seed_values)) if len(seed_values) > 1 else np.array([0.0])
                    gain_ax.scatter(
                        seed_values, y_position + offset + jitter, s=26, color=color,
                        alpha=0.4, edgecolors='none', zorder=2,
                    )
                    gain_extent.extend(seed_values.tolist())
                if not gain_rows.empty:
                    gain_row = gain_rows.iloc[0]
                    gain_std = finite_std(gain_row, 'std_gain')
                    gain_ax.errorbar(
                        gain_row['mean_gain'], y_position + offset, xerr=gain_std,
                        fmt='D', markersize=7, color=color, markeredgecolor='black',
                        markeredgewidth=0.7, capsize=3, linewidth=1.8, zorder=4,
                    )
                    gain_extent.append(float(gain_row['mean_gain']))
                    if gain_std is not None:
                        gain_extent.extend([gain_row['mean_gain'] - gain_std, gain_row['mean_gain'] + gain_std])

        tick_labels = []
        for model in models:
            seeds = absolute_slice[
                absolute_slice['model'].eq(model) & absolute_slice['metric_family'].eq(metric_name)
            ]['seeds']
            tick_labels.append(f'{MODEL_LABELS.get(model, model)}  (n={int(seeds.max()) if not seeds.empty else 0})')
        for ax in (absolute_ax, gain_ax):
            ax.set_yticks(y_positions, tick_labels)
            ax.set_ylim(len(models) - 0.5, -0.5)
            ax.grid(axis='x', color='#d1d5db', linewidth=0.8, alpha=0.8)
            ax.spines[['top', 'right', 'left']].set_visible(False)
            ax.tick_params(axis='y', length=0)

        direction = 'higher is better' if higher_is_better else 'lower is better'
        absolute_unit = '%' if unit == 'percentage points' else 'raw AHD units'
        absolute_ax.set_title(f'{title} — absolute ({direction})', loc='left', fontsize=11)
        absolute_ax.set_xlabel(f'Mean test score ({absolute_unit})')

        finite_extent = [abs(float(value)) for value in gain_extent if pd.notna(value)]
        default_extent = 1.0 if unit == 'percentage points' else 0.05
        limit = 1.18 * max(max(finite_extent, default=0.0), default_extent)
        gain_ax.set_xlim(-limit, limit)
        gain_ax.axvspan(-limit, 0, color='#fef2f2', zorder=0)
        gain_ax.axvspan(0, limit, color='#f0fdf4', zorder=0)
        gain_ax.axvline(0, color='#374151', linewidth=1.1, zorder=1)
        gain_ax.text(0.02, 0.97, '← native better', transform=gain_ax.transAxes,
                     ha='left', va='top', fontsize=8, color='#991b1b')
        gain_ax.text(0.98, 0.97, 'this cell better →', transform=gain_ax.transAxes,
                     ha='right', va='top', fontsize=8, color='#166534')
        gain_ax.set_title(f'{title} — gain vs native', loc='left', fontsize=11)
        gain_ax.set_xlabel(f'Direction-adjusted gain ({unit})')

    dataset_label = DATASET_LABELS.get(dataset, dataset)
    checkpoint_note = (
        f'{decoder}-selected checkpoint' if MATCHED_CHECKPOINT else 'all checkpoints'
    )
    fig.suptitle(
        f'{dataset_label} · {DECODER_LABELS[decoder]} decoding · {checkpoint_note}',
        fontsize=15, fontweight='bold', y=0.995,
    )
    legend_handles = [
        Line2D([0], [0], marker='D', color=INFERENCE_COLORS[cell],
               markerfacecolor=INFERENCE_COLORS[cell], linewidth=0, markersize=7,
               label=INFERENCE_LABELS[cell])
        for cell in cells
    ]
    legend_handles.append(
        Line2D([0], [0], marker='o', color='#4b5563', markerfacecolor='white',
               linewidth=0, markersize=7, label='hollow = that model\'s native cell')
    )
    fig.legend(handles=legend_handles, loc='lower center', ncol=min(len(legend_handles), 5),
               frameon=False, bbox_to_anchor=(0.5, 0.012))
    fig.text(
        0.5, 0.0,
        'Large markers show mean ± sample SD; no SD bar is drawn for n=1. '
        'Small dots are individual seed gains. Gains are measured against each '
        'model\'s own native inference, so the native cell has no gain marker.',
        ha='center', va='bottom', fontsize=9, color='#4b5563',
    )
    fig.tight_layout(rect=(0.02, 0.075, 0.99, 0.955), h_pad=2.0, w_pad=2.5)

    if SAVE_PLOTS:
        PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        dataset_slug = dataset.lower().replace('-', '_')
        output_path = PLOT_OUTPUT_DIR / f'{dataset_slug}_{decoder}_inference_comparison.png'
        fig.savefig(output_path, dpi=180, bbox_inches='tight')
        print('[saved]', output_path)
    plt.show()
    plt.close(fig)


if not absolute_summary.empty:
    for dataset in sorted(set(absolute_summary['dataset'])):
        for decoder in ('topdown', 'independent'):
            if decoder in set(absolute_summary['decoder']):
                plot_inference_comparison(dataset, decoder, absolute_summary, gain_summary, paired)
else:
    print('Nothing to plot for the current selection.')

## Interpreting the hypothesis

Evidence is **consistent with inference being a transferable main advantage** only if both conditions hold:

1. `subspace_norm` gives positive gains over the native `node_score` cell for H-CAST and HRN, preferably across both decoders and multiple seeds.
2. `node_score` degrades Hier-COS relative to its native `subspace_norm` cell.

Alternative outcomes:

- If native Hier-COS degrades under `node_score` but H-CAST/HRN do not improve under `subspace_norm`, the readout likely depends on the Hier-COS-trained representation rather than transferring by itself.
- If H-CAST/HRN improve but Hier-COS does not degrade, subspace decoding is useful post hoc but does not explain Hier-COS's advantage.
- If gains vary strongly by dataset, decoder, or seed, report the dependence instead of claiming a general mechanism.

Reading the `hcc_` cells needs two structural facts, both properties of the rule rather than results:

- `hcc_node_score` shifts every sibling group by a single constant, so for a signed readout it **cannot change any top-down metric** — expect exactly zero gain there for H-CAST and HRN, and read only the independent-decoding row. It does move both decoders for Hier-COS, whose readout takes the coordinate's magnitude.
- `subspace_norm` squares its inputs and therefore discards the sign of a classifier logit. That is the substantive content of the identity-frame assumption, not a side effect.

Do not infer causality from absolute model rankings alone. The paired within-checkpoint gains are the relevant evidence.